# Watermelon photo features, step by step
This notebook turns watermelon photos into numbers for your sheet. Each section does **one step** on **one photo**, shows what the code found, then the last sections run it on every photo and build the table.

Run the cells top to bottom (Shift+Enter). Two things to know before you read the code:
* An image is a grid of pixels. Every pixel has three numbers. We convert them to **CIELAB** (L\*, a\*, b\*) because it separates *how bright* a pixel is (L\*) from *what colour* it is (a\*: green↔red, b\*: blue↔yellow). That makes "is this pixel yellow?" a simple `b > 38` test.
* All the thresholds live in one dictionary `T`, so you can change a number and re-run to see what happens.

## 0. Setup

In [ ]:
!pip -q install pillow-heif
import os, glob, io, math
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import pillow_heif; pillow_heif.register_heif_opener()
plt.rcParams['figure.dpi'] = 90
def show(img, title='', cmap=None, size=6):
    plt.figure(figsize=(size, size*img.shape[0]/img.shape[1])); plt.title(title); plt.axis('off')
    plt.imshow(img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB), cmap=cmap); plt.show()

## 1. Get the photos
**In Colab:** run the next cell, sign in and allow access. It mounts your Google Drive and looks for the melon folder (it searches for `IMG_9597.HEIC`). If it isn't found, open the shared folder in Drive, right-click → *Organize → Add shortcut* → *My Drive*, then run it again.

**Not in Colab:** set `RAW_DIR` to a folder that holds the photos.

In [ ]:
RAW_DIR = None          # set this by hand if you are not using Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if any(f.upper() == 'IMG_9597.HEIC' or f.upper() == 'IMG_9597.JPG' for f in files):
            RAW_DIR = root; break
except ImportError:
    pass
print('photo folder:', RAW_DIR)
assert RAW_DIR, 'Photo folder not found - see the text above'
print(len(os.listdir(RAW_DIR)), 'files')

## 2. Name each photo by melon
The phone names photos `IMG_xxxx`. Each melon has 1 ID photo (shows the sticker number) followed by 6 side photos. I matched every `IMG_` file to a melon by reading the sticker numbers, and the result is the table below: `01_v1.jpg` = melon 1, view 1; `01_id.jpg` = melon 1's ID photo. `IMG_9591` (the group photo) is not in the table, so it is ignored.

The conversion step also fixes **rotation** (phones store it in EXIF metadata) and **resizes to 2576 px wide**, the size every threshold in this notebook was tuned on.

In [ ]:
MAP_CSV = """drive_name,melon_file
IMG_9760.jpg,01_id.jpg
IMG_9753.jpg,02_id.jpg
IMG_9746.jpg,03_id.jpg
IMG_9739.jpg,04_id.jpg
IMG_9731.jpg,05_id.jpg
IMG_9725.jpg,06_id.jpg
IMG_9718.jpg,07_id.jpg
IMG_9710.jpg,08_id.jpg
IMG_9703.jpg,09_id.jpg
IMG_9696.jpg,10_id.jpg
IMG_9689.jpg,11_id.jpg
IMG_9682.jpg,12_id.jpg
IMG_9675.jpg,13_id.jpg
IMG_9668.jpg,14_id.jpg
IMG_9661.jpg,15_id.jpg
IMG_9654.jpg,16_id.jpg
IMG_9647.jpg,17_id.jpg
IMG_9639.jpg,18_id.jpg
IMG_9626.jpg,20_id.jpg
IMG_9619.jpg,21_id.jpg
IMG_9612.jpg,22_id.jpg
IMG_9605.jpg,23_id.jpg
IMG_9597.jpg,24_id.jpg
IMG_9767.jpg,25_id.jpg
IMG_9761.jpg,01_v1.jpg
IMG_9762.jpg,01_v2.jpg
IMG_9763.jpg,01_v3.jpg
IMG_9764.jpg,01_v4.jpg
IMG_9765.jpg,01_v5.jpg
IMG_9766.jpg,01_v6.jpg
IMG_9754.jpg,02_v1.jpg
IMG_9755.jpg,02_v2.jpg
IMG_9756.jpg,02_v3.jpg
IMG_9757.jpg,02_v4.jpg
IMG_9758.jpg,02_v5.jpg
IMG_9759.jpg,02_v6.jpg
IMG_9747.jpg,03_v1.jpg
IMG_9748.jpg,03_v2.jpg
IMG_9749.jpg,03_v3.jpg
IMG_9750.jpg,03_v4.jpg
IMG_9751.jpg,03_v5.jpg
IMG_9752.jpg,03_v6.jpg
IMG_9740.jpg,04_v1.jpg
IMG_9741.jpg,04_v2.jpg
IMG_9742.jpg,04_v3.jpg
IMG_9743.jpg,04_v4.jpg
IMG_9744.jpg,04_v5.jpg
IMG_9745.jpg,04_v6.jpg
IMG_9732.jpg,05_v1.jpg
IMG_9733.jpg,05_v2.jpg
IMG_9735.jpg,05_v3.jpg
IMG_9736.jpg,05_v4.jpg
IMG_9737.jpg,05_v5.jpg
IMG_9738.jpg,05_v6.jpg
IMG_9726.jpg,06_v1.jpg
IMG_9727.jpg,06_v2.jpg
IMG_9728.jpg,06_v3.jpg
IMG_9729.jpg,06_v4.jpg
IMG_9730.jpg,06_v5.jpg
IMG_9719.jpg,07_v1.jpg
IMG_9720.jpg,07_v2.jpg
IMG_9721.jpg,07_v3.jpg
IMG_9722.jpg,07_v4.jpg
IMG_9723.jpg,07_v5.jpg
IMG_9724.jpg,07_v6.jpg
IMG_9711.jpg,08_v1.jpg
IMG_9712.jpg,08_v2.jpg
IMG_9713.jpg,08_v3.jpg
IMG_9714.jpg,08_v4.jpg
IMG_9715.jpg,08_v5.jpg
IMG_9716.jpg,08_v6.jpg
IMG_9704.jpg,09_v1.jpg
IMG_9705.jpg,09_v2.jpg
IMG_9706.jpg,09_v3.jpg
IMG_9707.jpg,09_v4.jpg
IMG_9708.jpg,09_v5.jpg
IMG_9709.jpg,09_v6.jpg
IMG_9697.jpg,10_v1.jpg
IMG_9698.jpg,10_v2.jpg
IMG_9699.jpg,10_v3.jpg
IMG_9700.jpg,10_v4.jpg
IMG_9701.jpg,10_v5.jpg
IMG_9702.jpg,10_v6.jpg
IMG_9690.jpg,11_v1.jpg
IMG_9691.jpg,11_v2.jpg
IMG_9692.jpg,11_v3.jpg
IMG_9693.jpg,11_v4.jpg
IMG_9694.jpg,11_v5.jpg
IMG_9695.jpg,11_v6.jpg
IMG_9683.jpg,12_v1.jpg
IMG_9684.jpg,12_v2.jpg
IMG_9685.jpg,12_v3.jpg
IMG_9686.jpg,12_v4.jpg
IMG_9687.jpg,12_v5.jpg
IMG_9688.jpg,12_v6.jpg
IMG_9676.jpg,13_v1.jpg
IMG_9677.jpg,13_v2.jpg
IMG_9678.jpg,13_v3.jpg
IMG_9679.jpg,13_v4.jpg
IMG_9680.jpg,13_v5.jpg
IMG_9681.jpg,13_v6.jpg
IMG_9669.jpg,14_v1.jpg
IMG_9670.jpg,14_v2.jpg
IMG_9671.jpg,14_v3.jpg
IMG_9672.jpg,14_v4.jpg
IMG_9673.jpg,14_v5.jpg
IMG_9674.jpg,14_v6.jpg
IMG_9662.jpg,15_v1.jpg
IMG_9663.jpg,15_v2.jpg
IMG_9664.jpg,15_v3.jpg
IMG_9665.jpg,15_v4.jpg
IMG_9666.jpg,15_v5.jpg
IMG_9667.jpg,15_v6.jpg
IMG_9655.jpg,16_v1.jpg
IMG_9656.jpg,16_v2.jpg
IMG_9657.jpg,16_v3.jpg
IMG_9658.jpg,16_v4.jpg
IMG_9659.jpg,16_v5.jpg
IMG_9660.jpg,16_v6.jpg
IMG_9648.jpg,17_v1.jpg
IMG_9649.jpg,17_v2.jpg
IMG_9650.jpg,17_v3.jpg
IMG_9651.jpg,17_v4.jpg
IMG_9652.jpg,17_v5.jpg
IMG_9653.jpg,17_v6.jpg
IMG_9640.jpg,18_v1.jpg
IMG_9641.jpg,18_v2.jpg
IMG_9643.jpg,18_v3.jpg
IMG_9644.jpg,18_v4.jpg
IMG_9645.jpg,18_v5.jpg
IMG_9646.jpg,18_v6.jpg
IMG_9633.jpg,19_v1.jpg
IMG_9634.jpg,19_v2.jpg
IMG_9635.jpg,19_v3.jpg
IMG_9636.jpg,19_v4.jpg
IMG_9637.jpg,19_v5.jpg
IMG_9638.jpg,19_v6.jpg
IMG_9627.jpg,20_v1.jpg
IMG_9628.jpg,20_v2.jpg
IMG_9629.jpg,20_v3.jpg
IMG_9630.jpg,20_v4.jpg
IMG_9631.jpg,20_v5.jpg
IMG_9632.jpg,20_v6.jpg
IMG_9620.jpg,21_v1.jpg
IMG_9621.jpg,21_v2.jpg
IMG_9622.jpg,21_v3.jpg
IMG_9623.jpg,21_v4.jpg
IMG_9624.jpg,21_v5.jpg
IMG_9625.jpg,21_v6.jpg
IMG_9613.jpg,22_v1.jpg
IMG_9614.jpg,22_v2.jpg
IMG_9615.jpg,22_v3.jpg
IMG_9616.jpg,22_v4.jpg
IMG_9617.jpg,22_v5.jpg
IMG_9618.jpg,22_v6.jpg
IMG_9606.jpg,23_v1.jpg
IMG_9607.jpg,23_v2.jpg
IMG_9608.jpg,23_v3.jpg
IMG_9609.jpg,23_v4.jpg
IMG_9610.jpg,23_v5.jpg
IMG_9611.jpg,23_v6.jpg
IMG_9598.jpg,24_v1.jpg
IMG_9599.jpg,24_v2.jpg
IMG_9600.jpg,24_v3.jpg
IMG_9601.jpg,24_v4.jpg
IMG_9602.jpg,24_v5.jpg
IMG_9604.jpg,24_v6.jpg
IMG_9768.jpg,25_v1.jpg
IMG_9769.jpg,25_v2.jpg
IMG_9770.jpg,25_v3.jpg
IMG_9771.jpg,25_v4.jpg
IMG_9772.jpg,25_v5.jpg
IMG_9773.jpg,25_v6.jpg"""
mp = pd.read_csv(io.StringIO(MAP_CSV))
os.makedirs('photos', exist_ok=True)
by_stem = {os.path.splitext(f)[0].upper(): f for f in os.listdir(RAW_DIR)}
for _, r in mp.iterrows():
    stem = os.path.splitext(r.drive_name)[0].upper()
    out = os.path.join('photos', r.melon_file)
    if os.path.exists(out): continue
    im = ImageOps.exif_transpose(Image.open(os.path.join(RAW_DIR, by_stem[stem]))).convert('RGB')
    im = im.resize((2576, round(im.height*2576/im.width)), Image.LANCZOS)
    im.save(out, quality=92)
print(len(os.listdir('photos')), 'photos ready')

## 3. Look at one photo, and convert to L\*a\*b\*
`cv2.imread` gives the picture as an array `[row, column, (B,G,R)]`. We work on a **half-size copy** (faster; the bee-sting step later uses the full-size one). Then we convert to Lab.

OpenCV stores 8-bit Lab as 0-255 for every channel, so `lab_channels` rescales to the usual ranges: L\* 0-100, and a\*, b\* centred on 0.

In [ ]:
def read(path, scale=0.5):
    im = cv2.imread(path)
    return im, cv2.resize(im, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

def lab_channels(im):
    lab = cv2.cvtColor(im, cv2.COLOR_BGR2LAB).astype(np.float32)
    return lab[..., 0] * 100 / 255, lab[..., 1] - 128, lab[..., 2] - 128

full, im = read('photos/15_v1.jpg')
L, a, b = lab_channels(im)
print('full size', full.shape, ' half size', im.shape)
show(im, 'melon 15, view 1')
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for x, name, cmap in [(L, 'L* (brightness)', 'gray'), (a, 'a* (green <-> red)', 'RdYlGn_r'), (b, 'b* (blue <-> yellow)', 'YlOrBr')]:
    h = ax[['L','a','b'].index(name[0])].imshow(x, cmap=cmap); ax[['L','a','b'].index(name[0])].set_title(name); plt.colorbar(h, ax=ax[['L','a','b'].index(name[0])])
plt.show()

Look at the **b\*** panel: the yellow field spot lights up, which is why a simple threshold on b\* will find it.

## 4. Thresholds
Every number the detector uses. Change one and re-run the steps below to see how it behaves.

In [ ]:
T = dict(
    web_a_min=-2.5, web_b_min=8,   # webbing: not green (a* above this) and some yellow-brown colour (b* above this)
    mask_a=-2.5,          # melon skin is greenish: a* below this
    mask_dark=26,         # or very dark (shaded side): L* below this
    spot_b=38,            # field spot: strongly yellow (b* above this) ...
    spot_L=42,            # ... and light
    spot_min_frac=0.004,  # spot must cover at least 0.4% of the rind to count
    pale_chroma=22,       # webbing + gloss are 'low colour intensity' ...
    pale_hue=108,         # ... and less green than the rind
    gloss_L=46,           # gloss = the brightest of those pale pixels
    stripe_light_b=17,    # light-green stripe: b* above this (after blur)
    stripe_min_px=6,      # ignore runs shorter than this
    bee_blackhat=22,      # bee sting: small dark dot, darker than its surroundings by this much ...
    bee_L=26, bee_a=-4.5, bee_b=10,   # ... and brown/black rather than dark green
    bee_area=(6, 500),    # size in full-resolution pixels
    side_min_ratio=1.07,  # ellipse axis ratio below this = looking at the stem/blossom end
)

## 5. Find the melon (segmentation)
Goal: a mask that is 1 on the melon and 0 on the floor and wall.
1. Keep pixels that are green (`a < -2.5`) and not too bright, **or** very dark (the shaded side of dark melons).
2. `MORPH_OPEN` removes small specks. `connectedComponentsWithStats` finds separate blobs and we keep the biggest one.
3. `MORPH_CLOSE` fills holes (spots, stickers that were not green).
4. A melon has no dents, so the **convex hull** (a rubber band around the shape) gives a clean outline.

In [ ]:
def segment(L, a, b):
    m = (((a < T['mask_a']) & (L < 75)) | (L < T['mask_dark'])).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((9, 9), np.uint8))
    n, lbl, st, _ = cv2.connectedComponentsWithStats(m)
    k = 1 + np.argmax(st[1:, cv2.CC_STAT_AREA]); m = (lbl == k).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((25, 25), np.uint8))
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    hull = cv2.convexHull(max(cnts, key=cv2.contourArea))      # melon is convex
    out = np.zeros_like(m); cv2.drawContours(out, [hull], -1, 1, -1)
    return out, hull

mask, hull = segment(L, a, b)
vis = im.copy(); cv2.drawContours(vis, [hull], -1, (255, 255, 255), 3)
show(vis, 'white line = melon outline found')
print('melon pixels:', int(mask.sum()))

## 6. Price sticker
Some melons carry a sticker whose colours would be mistaken for a field spot. Its rim is **orange** (a\* > 5), which a melon never is, so we start from orange pixels, grow outward over the sticker's yellow/blue/red print, fill the outline, and exclude it. Melon 13 view 3 has one.

In [ ]:
def find_sticker(L, a, b, mask):
    inside = mask > 0
    seed = ((a > 5) & (b > 20) & (L > 45) & inside).astype(np.uint8)
    seed = cv2.morphologyEx(seed, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    if seed.sum() < 300:
        return np.zeros_like(seed)
    print_ = (((b > 38) & (L > 45)) | ((b < -5) & (L > 50)) | (a > 5)) & inside
    grow = cv2.morphologyEx(print_.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((21, 21), np.uint8))
    n, lbl, st, _ = cv2.connectedComponentsWithStats(grow)
    out = np.zeros_like(seed)
    for i in np.unique(lbl[seed > 0]):
        if i == 0: continue
        pts = cv2.findNonZero((lbl == i).astype(np.uint8))
        cv2.drawContours(out, [cv2.convexHull(pts)], -1, 1, -1)
    return cv2.dilate(out, np.ones((9, 9), np.uint8))

f13, im13 = read('photos/13_v3.jpg'); L13, a13, b13 = lab_channels(im13)
m13, h13 = segment(L13, a13, b13); st13 = find_sticker(L13, a13, b13, m13)
v = im13.copy(); v[st13 > 0] = (255, 0, 255); show(cv2.addWeighted(v, .6, im13, .4, 0), 'purple = sticker, ignored from now on')

## 7. Rind area, view type and width-to-height ratio
* **Rind** = the mask shrunk a little (`erode`, so edge pixels that blend with the floor are dropped) minus the sticker. All percentages below are "% of rind pixels".
* `cv2.fitEllipse` fits an ellipse to the outline. **long axis ÷ short axis** is the width-to-height ratio you asked for (1.0 = round).
* If that ratio is under 1.07 the melon is being seen end-on (stem or blossom end), so it is flagged `end` and skipped for stripe width.

In [ ]:
sticker = find_sticker(L, a, b, mask)
rind = cv2.erode(mask, np.ones((15, 15), np.uint8)); rind[sticker > 0] = 0
(cx, cy), (ea, eb), ang = cv2.fitEllipse(hull)
major, minor = max(ea, eb), min(ea, eb)
x, y, w, h = cv2.boundingRect(hull)
print('rind pixels:', int(rind.sum()))
print(f'ellipse axes {major:.0f} x {minor:.0f} px  ->  ratio {major/minor:.3f}   (view type: {"side" if major/minor >= T["side_min_ratio"] else "end"})')
print(f'bounding box ratio w/h = {w/h:.3f}')
vis = im.copy(); cv2.ellipse(vis, ((cx, cy), (ea, eb), ang), (0, 255, 255), 3); cv2.rectangle(vis, (x, y), (x+w, y+h), (255, 255, 255), 2)
show(vis, 'yellow ellipse fitted to the outline, white = bounding box')

## 8. Field spot
A field spot is a **light, strongly yellow** patch: `b* > 38` and `L* > 42`, inside the rind. Then:
* `MORPH_OPEN` removes isolated yellow pixels.
* Blobs smaller than 0.4% of the rind are dropped (so the spot is a patch, not noise).
* The spot's colour is the **average L\*, a\*, b\*** of its pixels. These are the three colour columns in the sheet.

In [ ]:
def find_spot(L, a, b, rind):
    rind_px = int(rind.sum())
    spot = ((b > T['spot_b']) & (L > T['spot_L']) & (rind > 0)).astype(np.uint8)
    spot = cv2.morphologyEx(spot, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))
    n, lbl, st, _ = cv2.connectedComponentsWithStats(spot)
    keep = np.zeros_like(spot)
    for i in range(1, n):
        if st[i, cv2.CC_STAT_AREA] >= T['spot_min_frac'] * rind_px: keep[lbl == i] = 1
    return keep

spot = find_spot(L, a, b, rind)
present = spot.sum() > 0
print('spot area = %.2f%% of rind' % (100 * spot.sum() / rind.sum()))
if present:
    print('colour:  L* %.1f   a* %.1f   b* %.1f' % (L[spot > 0].mean(), a[spot > 0].mean(), b[spot > 0].mean()))
v = im.copy(); v[spot > 0] = (0, 230, 255); show(cv2.addWeighted(v, .6, im, .4, 0), 'yellow = detected field spot')

**Try it:** change `spot_b` in the thresholds cell to 30 and re-run. The detected area grows, because paler yellow now counts.

## 9. Gloss and webbing
Both are "not the normal green rind", so we first find **pale pixels**: low colour intensity (chroma `C* = sqrt(a² + b²)`) and a hue angle that is less green than the stripes.
* **Gloss**: the *brightest* pale pixels (blurred L\* ≥ 46) are the specular highlights where light reflects off the shiny rind. `rind gloss` is the % of rind pixels in that set.
* **Webbing**: tan/brown corky scar tissue. Its colour has a\* near 0 or above (not green) and some yellow-brown (b\* from 8 up to the spot threshold), not too dark, and **not near a highlight** (glare also looks pale).

In [ ]:
def gloss_webbing(L, a, b, rind, spot):
    C = np.hypot(a, b); hue = np.degrees(np.arctan2(b, a))
    pale = (C < T['pale_chroma']) & (hue < T['pale_hue']) & (rind > 0) & (spot == 0)
    Ls = cv2.GaussianBlur(L, (0, 0), 2)
    gloss = (pale & (Ls >= T['gloss_L'])).astype(np.uint8)
    gloss = cv2.morphologyEx(gloss, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    near_gloss = cv2.dilate(gloss, np.ones((25, 25), np.uint8))
    webbing = ((a > T['web_a_min']) & (b > T['web_b_min']) & (b < T['spot_b']) & (L >= 38)
               & (rind > 0) & (spot == 0) & (near_gloss == 0)).astype(np.uint8)
    webbing = cv2.morphologyEx(webbing, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    return gloss, webbing

fm, imm = read('photos/23_v4.jpg'); Lm, am, bm = lab_channels(imm)
mk, hl = segment(Lm, am, bm); stk = find_sticker(Lm, am, bm, mk)
rd = cv2.erode(mk, np.ones((15, 15), np.uint8)); rd[stk > 0] = 0
sp = find_spot(Lm, am, bm, rd); gl, wb = gloss_webbing(Lm, am, bm, rd, sp)
print('gloss %.2f%%   webbing %.2f%%' % (100*gl.sum()/rd.sum(), 100*wb.sum()/rd.sum()))
v = imm.copy(); v[wb > 0] = (0, 0, 255); v[gl > 0] = (255, 200, 0); show(cv2.addWeighted(v, .6, imm, .4, 0), 'red = webbing, cyan = gloss (melon 23)')

These are the least exact measures: gloss depends on where the light was, and webbing can pick up the odd scratch. Always glance at the overlay before trusting a number.

## 10. Stripe width (striped melons only)
Stripes run roughly horizontally in side views, so we walk **down vertical lines** through the middle of the melon and see how the colour changes: light-green stripe (b\* above 17 after blur) → dark stripe → light → …
* Each unbroken run of "dark" pixels is one dark stripe; its length in pixels is the stripe width.
* We skip the top and bottom 20% (stripes bunch together there), lines that cross the sticker or edge, and the first/last run of each line (cut off by the window).
* The reported number is `mean dark run / melon height`, so it does not depend on how big the melon looks in the photo.

In [ ]:
def stripe_width(L, a, b, rind, hull_box):
    x, y, w, h = hull_box
    bb = cv2.GaussianBlur(b, (0, 0), 3)
    light = (bb > T['stripe_light_b'])
    y0, y1 = int(y + 0.2 * h), int(y + 0.8 * h)
    dark_runs, light_runs, n_dark = [], [], []
    for cx in range(int(x + 0.25 * w), int(x + 0.75 * w), 4):
        col_ok = rind[y0:y1, cx] > 0
        if col_ok.mean() < 0.95: continue
        s = light[y0:y1, cx]
        edges = np.flatnonzero(np.diff(s.astype(np.int8))) + 1
        bounds = np.r_[0, edges, len(s)]
        runs = [(bounds[i], bounds[i + 1] - bounds[i], bool(s[bounds[i]])) for i in range(len(bounds) - 1)]
        runs = [r for r in runs if r[1] >= T['stripe_min_px']]
        inner = runs[1:-1]
        dark_runs += [r[1] for r in inner if not r[2]]
        light_runs += [r[1] for r in inner if r[2]]
        n_dark.append(sum(1 for r in runs if not r[2]))
    if not dark_runs: return np.nan, np.nan, np.nan
    return np.mean(dark_runs) / h, np.mean(light_runs) / h if light_runs else np.nan, float(np.median(n_dark))

fs, ims = read('photos/25_v3.jpg'); Ls_, as_, bs_ = lab_channels(ims)
ms, hs = segment(Ls_, as_, bs_); rs = cv2.erode(ms, np.ones((15, 15), np.uint8))
box = cv2.boundingRect(hs); sw = stripe_width(Ls_, as_, bs_, rs, box)
print('dark stripe width / height = %.3f   light = %.3f   dark stripes per scan line = %.0f' % sw)
# draw what one scan line sees
xx, yy, ww, hh = box; cx0 = int(xx + 0.5*ww); y0, y1 = int(yy+.2*hh), int(yy+.8*hh)
lightmask = cv2.GaussianBlur(bs_, (0, 0), 3) > T['stripe_light_b']
fig, ax = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={'width_ratios': [3, 1]})
v = ims.copy(); cv2.line(v, (cx0, y0), (cx0, y1), (0, 0, 255), 3); ax[0].imshow(cv2.cvtColor(v, cv2.COLOR_BGR2RGB)); ax[0].axis('off'); ax[0].set_title('red line = one scan line')
ax[1].plot(lightmask[y0:y1, cx0].astype(int), range(y1-y0)); ax[1].invert_yaxis(); ax[1].set_title('1 = light stripe, 0 = dark')
plt.show()

## 11. Bee stings
A bee sting is a **small, dark, round, isolated** brown dot. This step runs on the **full-size** image because the dots are only a few pixels wide.
1. **Black-hat** (`MORPH_BLACKHAT`) highlights spots that are darker than their surroundings. This ignores the overall dark of the melon and finds local dark dots.
2. Keep pixels that are strongly dark-spot AND brown/black rather than dark green (`L`, `a`, `b` limits), away from the melon edge and the sticker.
3. Keep blobs of the right size, roughly round (not long scratches or hairs).
4. A real sting is **isolated**: dots with another candidate within 80 px are speckle texture, so they are not counted.

In [ ]:
def bee_stings(im_full, rind_full, sticker_full):
    L, a, b = lab_channels(im_full)
    Lb = cv2.GaussianBlur(L, (0, 0), 1.2)
    bh = cv2.morphologyEx(Lb, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41)))
    inner = cv2.erode(rind_full, np.ones((61, 61), np.uint8))
    cand = ((bh > T['bee_blackhat']) & (Lb < T['bee_L']) & (a > T['bee_a']) & (b < T['bee_b'])
            & (inner > 0) & (sticker_full == 0)).astype(np.uint8)
    n, lbl, st, cen = cv2.connectedComponentsWithStats(cand)
    pts = []
    for i in range(1, n):
        area = st[i, cv2.CC_STAT_AREA]; bw, bhh = st[i, cv2.CC_STAT_WIDTH], st[i, cv2.CC_STAT_HEIGHT]
        if not (T['bee_area'][0] <= area <= T['bee_area'][1]): continue
        if max(bw, bhh) / max(1, min(bw, bhh)) > 2.2: continue
        if area / (bw * bhh) < 0.45: continue
        pts.append((int(cen[i][0]), int(cen[i][1]), int(area)))
    iso = []
    for i, (x, y, ar) in enumerate(pts):
        if all(np.hypot(x - x2, y - y2) >= 80 for j, (x2, y2, _) in enumerate(pts) if j != i):
            iso.append((x, y, ar))
    return pts, iso

fb, imb = read('photos/01_v1.jpg'); Lb_, ab_, bb_ = lab_channels(imb)
mb, hb = segment(Lb_, ab_, bb_); sb = find_sticker(Lb_, ab_, bb_, mb)
up = lambda m: cv2.resize(m, (fb.shape[1], fb.shape[0]), interpolation=cv2.INTER_NEAREST)
cands, stings = bee_stings(fb, up(mb), up(sb))
print(len(cands), 'candidate dots ->', len(stings), 'counted as bee stings after the isolation rule')
v = fb.copy()
for (px, py, ar) in cands: cv2.circle(v, (px, py), 28, (0, 255, 255), 3)
for (px, py, ar) in stings: cv2.circle(v, (px, py), 36, (255, 0, 255), 5)
show(v, 'yellow rings = candidates, magenta = counted', size=8)

## 12. Everything in one function
`process()` runs steps 3-11 on a photo and returns one row of numbers, plus the layers for the overlay picture. `overlay()` paints what was found so you can check the numbers by eye.

In [ ]:
def process(path):
    full, im = read(path, 0.5)
    L, a, b = lab_channels(im)
    mask, hull = segment(L, a, b)
    sticker = find_sticker(L, a, b, mask)
    rind = cv2.erode(mask, np.ones((15, 15), np.uint8)); rind[sticker > 0] = 0
    rind_px = int(rind.sum())
    (cx, cy), (ea, eb), ang = cv2.fitEllipse(hull)
    major, minor = max(ea, eb), min(ea, eb)
    x, y, w, h = cv2.boundingRect(hull)
    side_view = (major / minor) >= T['side_min_ratio']
    spot = find_spot(L, a, b, rind); spot_present = bool(spot.sum() > 0)
    gloss, webbing = gloss_webbing(L, a, b, rind, spot)
    sw = (np.nan, np.nan, np.nan)
    if side_view:
        plain = rind.copy(); plain[cv2.dilate(spot, np.ones((15, 15), np.uint8)) > 0] = 0
        sw = stripe_width(L, a, b, plain, (x, y, w, h))
    pts = bee_stings(full, cv2.resize(mask, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST),
                     cv2.resize(sticker, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST))[1]
    row = dict(file=os.path.basename(path), view_type='side' if side_view else 'end', rind_px=rind_px,
        spot_area_pct=100 * spot.sum() / rind_px,
        spot_L=float(L[spot > 0].mean()) if spot_present else np.nan,
        spot_a=float(a[spot > 0].mean()) if spot_present else np.nan,
        spot_b=float(b[spot > 0].mean()) if spot_present else np.nan,
        webbing_pct=100 * webbing.sum() / rind_px, gloss_pct=100 * gloss.sum() / rind_px,
        dark_stripe_width_over_height=sw[0], dark_stripes_per_scanline=sw[2],
        axis_ratio=major / minor, bee_sting_count=len(pts))
    return row, dict(im=im, hull=hull, spot=spot, webbing=webbing, gloss=gloss, sticker=sticker, pts=pts)

def overlay(layers):
    im = layers['im'].copy(); ov = im.copy()
    ov[layers['spot'] > 0] = (0, 230, 255); ov[layers['webbing'] > 0] = (0, 0, 255)
    ov[layers['gloss'] > 0] = (255, 200, 0); ov[layers['sticker'] > 0] = (255, 0, 255)
    im = cv2.addWeighted(ov, 0.55, im, 0.45, 0); cv2.drawContours(im, [layers['hull']], -1, (255, 255, 255), 2)
    for (px, py, ar) in layers['pts']: cv2.circle(im, (px // 2, py // 2), 14, (255, 0, 255), 2)
    return im

row, layers = process('photos/15_v1.jpg'); print(row)
show(overlay(layers), 'yellow = spot, red = webbing, cyan = gloss, purple = sticker, rings = bee stings')

## 13. Run on all photos
Takes a few minutes (each photo is processed at full size for the bee-sting step). The ID photos (`*_id.jpg`) are not used. The result is one row per photo.

In [ ]:
rows, ovs = [], {}
for f in sorted(glob.glob('photos/*_v[1-6].jpg')):
    r, lay = process(f); r['melon_id'] = int(os.path.basename(f).split('_')[0]); rows.append(r)
    ovs[os.path.basename(f)] = overlay(lay)
per_photo = pd.DataFrame(rows)
per_photo.round(2).head(12)

In [ ]:
# look at all six photos of any melon
def melon_sheet(n):
    tiles = [cv2.resize(ovs[f'{n:02d}_v{k}.jpg'], (640, 480)) for k in range(1, 7) if f'{n:02d}_v{k}.jpg' in ovs]
    show(np.vstack([np.hstack(tiles[:3]), np.hstack((tiles[3:] + [np.zeros_like(tiles[0])] * 3)[:3])]), f'melon {n}', size=12)
melon_sheet(5)

## 14. One value per melon
Per melon (6 photos):
* **field spot**: the biggest spot area seen in any photo (the underside photo shows it best). Colour = mean L\*a\*b\* of photos where a spot was found.
* **webbing, rind gloss, bee stings**: average over the photos.
* **stripe width, shape uniformity (width ÷ height)**: median over side views only.
* **Excluded photos**: three had failed outlines (04_v3 was blurry with a hand in frame; 23_v2 and 24_v2 were end-on shots where the outline was wrong). Edit `BAD` to change this.
* Stripe width is blanked for kinnaree melons, which are plain dark with no stripes.

In [ ]:
BAD = {'04_v3.jpg', '23_v2.jpg', '24_v2.jpg'}
SHEET_CSV = """no,type,Brix top,Brix middle,Brix bottom,Name,Brix sum
1,sonya,6.2,7.8,6.0,Makro,6.666666667
2,sonya,5.6,8.9,8.0,Aro,7.5
3,sonya,8.0,6.8,6.4,Zhonya,7.066666667
4,sonya,7.9,9.6,8.0,freddy,8.5
5,sonya,6.9,7.2,5.4,Markiplier,6.5
6,sonya,9.1,10.6,10.8,IshowSpeed,10.166666667
7,sonya,6.2,8.0,6.0,Claude,6.733333333
8,sonya,7.8,11.0,8.6,king,9.133333333
9,sonya,8.2,10.1,9.2,luther,9.166666667
10,sonya,6.4,7.8,7.2,martin,7.133333333
11,kinnaree,7.0,10.8,10.8,Kayn,9.533333333
12,kinnaree,6.9,11.2,7.8,Jayce,8.633333333
13,kinnaree,7.2,11.0,8.0,Tim,8.733333333
14,kinnaree,6.0,8.0,6.9,michael,6.966666667
15,kinnaree,6.0,9.0,7.0,Andre,7.333333333
16,kinnaree,6.4,8.5,7.0,Jyn,7.3
17,kinnaree,8.0,10.0,9.2,Pyke,9.066666667
18,kinnaree,6.0,8.4,5.4,Andy,6.6
19,kinnaree,8.0,11.0,8.2,Surapraween,9.066666667
20,kinnaree,8.4,11.3,9.0,Sasha,9.566666667
21,kinnaree,9.2,7.9,7.2,banner,8.1
22,kinnaree,8.9,10.0,8.2,Joseph,9.033333333
23,kinnaree,9.2,10.1,7.7,Tasque,9.0
24,kinnaree,8.4,9.8,8.6,Mike,8.933333333
25,sonya,3.8,7.0,6.0,Fridge,5.6"""
sheet = pd.read_csv(io.StringIO(SHEET_CSV))
d = per_photo[~per_photo.file.isin(BAD)]
side = d[d.view_type == 'side']
out = []
for m, g in d.groupby('melon_id'):
    gs = side[side.melon_id == m]; sp = g[g.spot_area_pct >= 0.4]
    out.append({'no': m,
        'field spot': round(g.spot_area_pct.max(), 2),
        'webbing': round(g.webbing_pct.mean(), 2),
        'rind gloss': round(g.gloss_pct.mean(), 2),
        'stripe width': round(gs.dark_stripe_width_over_height.median(), 2),
        'shape uniformity': round(gs.axis_ratio.median(), 2),
        'Bee stings': round(g.bee_sting_count.mean(), 2),
        'field spot colour (b*)': round(sp.spot_b.mean(), 2),
        'field spot L*': round(sp.spot_L.mean(), 2),
        'field spot a*': round(sp.spot_a.mean(), 2)})
feat = pd.DataFrame(out).merge(sheet, on='no')
feat.loc[feat.type != 'sonya', 'stripe width'] = np.nan
feat.to_csv('watermelon_features.csv', index=False)
feat

## 15. Do the features line up with sweetness?
A first look: how strongly does each feature move with **Brix sum** (the average of the three refractometer readings)? Correlation runs from −1 to +1; near 0 means no linear relationship. With only 25 melons, treat anything under about ±0.4 as noise.

In [ ]:
cols = ['field spot', 'webbing', 'rind gloss', 'stripe width', 'shape uniformity', 'Bee stings', 'field spot colour (b*)']
print(feat[cols + ['Brix sum']].corr()['Brix sum'].drop('Brix sum').round(2))

## Limits to remember
* Photos had no colour card, and lighting varied, so colour and gloss are only comparable roughly.
* Bee-sting counts include some false positives from speckled rind.
* Melon 6 has only 3 usable side views; melon 19 had no ID photo, so its identity was inferred.
* `knock sounds` still needs to come from your friend.